In [ ]:
pip install xgboost

In [ ]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO ÚNICO
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota"
mlflow.set_experiment(experiment_name)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Separar datos supervisados vs inferencia
df_supervised = df[df['v_ref_2026'].notnull()].copy()
df_inference = df[df['v_ref_2026'].isnull()].copy()

# Preparar Matriz X e y 
y_sup = df_supervised['v_ref_2026']
cols_a_excluir = ['barrio', 'geometry', 'v_ref_2015', 'v_ref_2026', 'num_listings_airbnb', 'area_km2']
X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])

# XGBoost requiere variables numéricas (aplicamos get_dummies como en Random Forest)
X_sup = pd.get_dummies(X_sup, columns=['localidad'], drop_first=True, dtype=int)

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)

# Hiperparámetros iniciales
params = {
    "n_estimators": 1000,
    "learning_rate": 0.01,
    "max_depth": 3,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_lambda": 5,
    "random_state": 42,
    "objective": "reg:squarederror"
}

# ==============================================================================
# RUN 1: XGBOOST SIN LOGARITMO (Precios Brutos)
# ==============================================================================
with mlflow.start_run(run_name=f"XGB_Sin_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Ninguna")

    # Entrenamiento
    xgb_bruto = xgb.XGBRegressor(**params)
    xgb_bruto.fit(X_train, y_train)

    # Evaluación Train
    y_pred_train_bruto = xgb_bruto.predict(X_train)
    mae_train_bruto = mean_absolute_error(y_train, y_pred_train_bruto)
    r2_train_bruto = r2_score(y_train, y_pred_train_bruto)

    # Evaluación Test
    y_pred_test_bruto = xgb_bruto.predict(X_test)
    mae_test_bruto = mean_absolute_error(y_test, y_pred_test_bruto)
    rmse_test_bruto = np.sqrt(mean_squared_error(y_test, y_pred_test_bruto))
    r2_test_bruto = r2_score(y_test, y_pred_test_bruto)

    # Registro de métricas
    mlflow.log_metric("mae_train_pesos", mae_train_bruto)
    mlflow.log_metric("mae_test_pesos", mae_test_bruto)
    mlflow.log_metric("r2_train", r2_train_bruto)
    mlflow.log_metric("r2_test", r2_test_bruto)
    mlflow.log_metric("rmse_test_pesos", rmse_test_bruto)

    print(f"--- XGBOOST SIN LOGARITMO --- R2 Test: {r2_test_bruto:.4f} | MAE Test: ${mae_test_bruto:,.0f}")

    # Gráfico de importancia de variables (Artefacto en MLflow)
    # Filtramos el top 15 para que el gráfico no se sature por las localidades
    importancias = xgb_bruto.feature_importances_
    top_15_idx = np.argsort(importancias)[-15:]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(top_15_idx)), importancias[top_15_idx], color='teal')
    ax.set_yticks(range(len(top_15_idx)))
    ax.set_yticklabels(X_train.columns[top_15_idx])
    ax.set_title("Top 15 Variables Más Importantes - XGBoost (Bruto)")
    plt.tight_layout()
    mlflow.log_figure(fig, "importancia_variables_bruto.png")
    plt.close(fig) # Cierra la figura para no saturar la memoria

    mlflow.xgboost.log_model(xgb_bruto, "xgboost_model_bruto", input_example=X_train.head(1))

# ==============================================================================
# RUN 2: XGBOOST CON LOGARITMO 
# ==============================================================================
with mlflow.start_run(run_name=f"XGB_Con_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Logaritmo Natural")

    # Transformación
    y_train_log = np.log(y_train)

    # Entrenamiento
    xgb_log = xgb.XGBRegressor(**params)
    xgb_log.fit(X_train, y_train_log)

    # Evaluación Train (Revertida)
    y_pred_train_log = xgb_log.predict(X_train)
    y_pred_train_rev = np.exp(y_pred_train_log)
    mae_train_rev = mean_absolute_error(y_train, y_pred_train_rev)
    r2_train_rev = r2_score(y_train, y_pred_train_rev)

    # Evaluación Test (Revertida)
    y_pred_test_log = xgb_log.predict(X_test)
    y_pred_test_rev = np.exp(y_pred_test_log)
    mae_test_rev = mean_absolute_error(y_test, y_pred_test_rev)
    rmse_test_rev = np.sqrt(mean_squared_error(y_test, y_pred_test_rev))
    r2_test_rev = r2_score(y_test, y_pred_test_rev)

    # Registro de métricas
    mlflow.log_metric("mae_train_pesos", mae_train_rev)
    mlflow.log_metric("mae_test_pesos", mae_test_rev)
    mlflow.log_metric("r2_train", r2_train_rev)
    mlflow.log_metric("r2_test", r2_test_rev)
    mlflow.log_metric("rmse_test_pesos", rmse_test_rev)

    mlflow.xgboost.log_model(xgb_log, "xgboost_model_log", input_example=X_train.head(1))
